# Property Data Scraper for Nawy Real Estate Platform

## Overview

This notebook contains a complete automation workflow designed to extract residential property listings from the Nawy real estate platform and save them for analysis.

## Step-by-Step Breakdown

### **Setup and Initialization**

The process begins by importing necessary libraries including async browser automation tools, HTML parsing utilities, and data handling packages. A target URL is defined pointing to the property search page on Nawy where residential listings are displayed.

### **Browser Automation with Async Browser Tooling**

An asynchronous function launches a headless browser instance capable of rendering JavaScript dynamically. Since many modern websites load content through client-side scripts, this approach ensures we capture all properties that appear after scrolling rather than just the initial batch.

### **Infinite Scroll Simulation**

To access all available listings, the script simulates user scrolling behavior inside the container holding property cards:

*   The browser navigates to the target page and waits for the main listing container to load
*   It then scrolls to the bottom repeatedly until no new properties appear
*   A counter tracks consecutive iterations where the property count remains unchanged
*   Once five straight iterations show no new additions, the system assumes all content has loaded and stops scrolling
*   Five-second delays between scrolls ensure adequate time for content to render

### **HTML Extraction After Scrolling**

Once scrolling completes, the full rendered page source is captured as HTML string before closing the browser connection.

### **Data Parsing with HTML Parser**

The captured HTML is parsed using an element-tree style parser to traverse the DOM structure. Each property card is located by finding cover image div elements, then moving up to their parent anchor tags that represent individual property entries.

### **Information Extraction Fields**

For each property card found, the following details are extracted:

*   **URL Path**: Links to individual property detail pages
*   **Tag Type**: Labels indicating property category (Resale, etc.)
*   **Cover Image**: Primary photo URL for the listing
*   **Location Area**: Geographic area or district name
*   **Property Name**: Official compound or project title
*   **Developer Logo**: Branding image associated with the developer
*   **Title and Description**: Full property name and brief description text
*   **Property Features**: Area size, number of bedrooms, bathrooms
*   **Payment Plan**: Down payment terms if specified
*   **Price**: Listed sale price when available

### **Data Output and Storage**

All extracted property dictionaries are collected into a list, converted into tabular format using spreadsheet software library, and saved as a comma-separated values file named Villa_nawy_properties.csv. The first few rows can be previewed directly within the notebook interface for verification before saving.

### **Key Considerations**

*   This scraper handles dynamic content requiring JavaScript execution
*   Headless mode enables deployment on servers or cloud environments without display
*   Multiple wait intervals prevent timing issues during rapid scrolling
*   Error checking ensures graceful handling when certain fields are missing
*   The CSV export makes results ready for further analysis in Excel, pandas, or BI tools

In [9]:
import requests
import json
from bs4 import BeautifulSoup

In [3]:
# Define the target URL for the Nawy property search page (page 1)
url = "https://www.nawy.com/search?page_number=1&category=property"
# Make an initial GET request to the website (not used in final scraping but kept for reference)
response = requests.get(url)

In [6]:
if response.status_code == 200:
    html_content = response.text
    # print(html_content)
else:
    print("Request failed:", response.status_code)

In [14]:
# Initialize BeautifulSoup
soup = BeautifulSoup(html_content, 'html.parser')

# Dictionary to hold the extracted data
extracted_data = {}

# 1. URL Path
# We find a distinct element of the property card (like the cover image)
# and get its parent <a> tag to avoid grabbing the site's header logo.
property_card_inner = soup.find('div', class_='cover-image')
a_tag = property_card_inner.find_parent('a') if property_card_inner else None

if a_tag and 'href' in a_tag.attrs:
    # Prepend the base URL to make it an absolute link
    extracted_data['url_path'] = f"https://www.nawy.com{a_tag['href']}"
else:
    extracted_data['url_path'] = None

# 2. Tag (e.g., "Resale")
tag_elem = soup.find('div', class_='tag')
extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

# 3. Cover Image URL
cover_image = soup.find('img', alt='Cover Image')
extracted_data['cover_image'] = cover_image['src'] if cover_image else None

# 4. Location
area_elem = soup.find('div', class_='area')
extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

# 5. Property Name & Compound
name_elem = soup.find('div', class_='name')
extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

# 6. Developer Logo URL
logo_image = soup.find('img', alt='logo')
extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

# 7. Full Title/Description
title_elem = soup.find('h2')
extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

# Description
desc_elem = soup.select_one('h2.sc-4b9910fd-0.hHfZHY')
extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

# 8. Property Features (Area, Beds, Baths)
values = soup.find_all('span', class_='value')
labels = soup.find_all('span', class_='label')

for val, lbl in zip(values, labels):
    feature_name = lbl.get_text(strip=True)
    feature_value = val.get_text(strip=True)
    extracted_data[feature_name] = feature_value

# 9. Payment Plan
# Using " ".join(...split()) helps remove heavy whitespace/newlines from the raw HTML text
down_payment_elem = soup.find('span', class_='down-payment')
if down_payment_elem:
    raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
    extracted_data['payment_plan'] = " ".join(raw_payment.split())
else:
    extracted_data['payment_plan'] = None

# 10. Price
price_elem = soup.find('span', class_='price')
if price_elem:
    raw_price = price_elem.get_text(separator=' ', strip=True)
    extracted_data['price'] = " ".join(raw_price.split())
else:
    extracted_data['price'] = None

# Print out the results formatted as pretty JSON
print(json.dumps(extracted_data, indent=4))

{
    "url_path": "https://www.nawy.com/compound/274-zed/property/117550-studio-for-sale-in-zed-with-1-bedrooms-in-el-sheikh-zayed-by-ora-developers",
    "tag": "Resale",
    "cover_image": "https://prod-images.cooingestate.com/processed/inventory/unlockedProperties/117550/25fb3f10-4de3-4ecf-a7d3-39e6d3f9b1b3/high.webp",
    "location": "El Sheikh Zayed",
    "property_name": "Studio, ZED",
    "developer_logo": "https://prod-images.cooingestate.com/processed/developer/logo_image/85/medium.webp",
    "title": "Nawy Now",
    "description": "Studio for sale in ZED  - with 1 bedroom in El Sheikh Zayed by Ora Developers.",
    "m2": "115",
    "Beds": "2",
    "Baths": "2",
    "payment_plan": "48,913 EGP quarterly / 4.6 Years",
    "price": "5,000,000 EGP"
}


In [41]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json

# Define the target URL for the Nawy property search page
url = "https://www.nawy.com/search?page_number=1&category=property"

async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container multiple times to load all properties
        for i in range(5):  # Changed from 100 to 10 for much faster testing
            print(f"Scrolling iteration {i+1}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        container.scrollBy(0, 1500);  // Scroll down by 1500px each time
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Reduced wait time from 5s to 2s

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Print out the results formatted as pretty JSON array
print(json.dumps(all_properties_data, indent=4))

Scrolling iteration 1...
Scrolling iteration 2...
Scrolling iteration 3...
Scrolling iteration 4...
Scrolling iteration 5...
[
    {
        "url_path": "https://www.nawy.com/compound/658-golden-gate/property/117594-office-for-sale-in-golden-gate-in-el-lotus-by-redcon-for-offices-and-commercial-centers",
        "tag": "Resale",
        "cover_image": "https://prod-images.cooingestate.com/processed/inventory/unlockedProperties/117594/63fcaa80-fcb0-4598-aef6-ee6f3a5e2707/high.webp",
        "location": "El Lotus",
        "property_name": "Office, Golden Gate",
        "developer_logo": "https://prod-images.cooingestate.com/processed/developer/logo_image/196/medium.webp",
        "title": "An office in Golden Gate by Redcon for Offices and Commercial Centers. The office size is 68 m2.",
        "description": "An office in Golden Gate by Redcon for Offices and Commercial Centers. The office size is 68 m2.",
        "m2": "68",
        "payment_plan": "592,956 EGP quarterly / 3.8 Years",

In [36]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
url = "https://www.nawy.com/search?page_number=1&category=property"

async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container multiple times to load all properties
        for i in range(5):  # Changed from 100 to 10 for much faster testing
            print(f"Scrolling iteration {i+1}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        container.scrollBy(0, 1500);  // Scroll down by 1500px each time
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Reduced wait time from 5s to 2s

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('nawy_properties.csv', index=False)
print("\nData successfully saved to nawy_properties.csv")

Scrolling iteration 1...
Scrolling iteration 2...
Scrolling iteration 3...
Scrolling iteration 4...
Scrolling iteration 5...

Data successfully saved to nawy_properties.csv


In [37]:
df.head()

,url_path,tag,cover_image,location,property_name,developer_logo,title,description,m2,Beds,Baths,payment_plan,price
0,https://www.nawy.com/compound/939-telal-east/p...,Resale,https://prod-images.cooingestate.com/processed...,New Cairo,"Duplex, Telal East",https://prod-images.cooingestate.com/processed...,Duplex for sale in Telal East - with 4 bedroom...,Duplex for sale in Telal East - with 4 bedroom...,233,4,3,"204,543 EGP quarterly / 5.4 Years","10,418,139 EGP"
1,https://www.nawy.com/compound/1364-noi/propert...,None,https://prod-images.cooingestate.com/processed...,6th settlement,"Villa, Noi",https://prod-images.cooingestate.com/processed...,Villa for sale in Noi - with 2 bedrooms in 6th...,Villa for sale in Noi - with 2 bedrooms in 6th...,191,2,4,"260,978 EGP monthly / 8 Years","25,053,857 EGP"
2,https://www.nawy.com/compound/1364-noi/propert...,None,https://prod-images.cooingestate.com/processed...,6th settlement,"Studio, Noi",https://prod-images.cooingestate.com/processed...,Studio for sale in Noi - with in 6th settlemen...,Studio for sale in Noi - with in 6th settlemen...,59,NaN,1,"241,509 EGP quarterly / 8 Years","7,728,292 EGP"
3,https://www.nawy.com/compound/422-mountain-vie...,Resale,https://prod-images.cooingestate.com/processed...,Northern Expansion,"Townhouse, Mountain View iCity October",https://prod-images.cooingestate.com/processed...,Townhouse for sale in Mountain View iCity Octo...,Townhouse for sale in Mountain View iCity Octo...,185,3,3,"431,250 EGP quarterly / 1 Years","13,425,000 EGP"
4,https://www.nawy.com/compound/1364-noi/propert...,None,https://prod-images.cooingestate.com/processed...,6th settlement,"Studio, Noi",https://prod-images.cooingestate.com/processed...,Studio for sale in Noi - with in 6th settlemen...,Studio for sale in Noi - with in 6th settlemen...,61,NaN,1,"249,696 EGP quarterly / 8 Years","7,990,268 EGP"


In [38]:
df.loc[3]['url_path']

'https://www.nawy.com/compound/422-mountain-view-icity-october/property/117580-townhouse-for-sale-in-mountain-view-icity-october-with-3-bedrooms-in-northern-expansion-by-mountain-view'

In [39]:
df.columns

Index(['url_path', 'tag', 'cover_image', 'location', 'property_name',
       'developer_logo', 'title', 'description', 'm2', 'Beds', 'Baths',
       'payment_plan', 'price'],
      dtype='object')

In [40]:
df.shape

(32, 13)

In [ ]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
url = "https://www.nawy.com/search?page_number=1&category=property"

async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container multiple times to load all properties
        for i in range(600):  # Changed from 100 to 10 for much faster testing
            print(f"Scrolling iteration {i+1}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        container.scrollBy(0, container.scrollHeight);  // Scroll down by 1500px each time, container.scrollHeight That removes all guessing about pixel distances.
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Reduced wait time from 5s to 2s

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('nawy_properties.csv', index=False)
print("\nData successfully saved to nawy_properties.csv")

In [55]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
# url = "https://www.nawy.com/search?page_number=1&category=property"
url = "https://www.nawy.com/search?category=property&property_types=21"


async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container until no new properties load for 3 iterations
        previous_count = 0
        iteration = 1
        unchanged_count_strikes = 0

        while True:
            print(f"Scrolling iteration {iteration}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        // Scroll to the bottom of the container
                        container.scrollTo(0, container.scrollHeight);
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Wait 5 seconds to allow content to load

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

            # Check if new items were loaded
            if current_count == previous_count:
                unchanged_count_strikes += 1
                print(f"No new properties loaded. Strike {unchanged_count_strikes}/3")
            else:
                unchanged_count_strikes = 0  # Reset the strike counter if new items appeared

            # Break the loop if no new items were loaded for 3 continuous iterations
            if unchanged_count_strikes >= 3:
                print("All properties loaded (count unchanged for 3 iterations). Stopping scroll.")
                break

            previous_count = current_count
            iteration += 1

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('nawy_properties.csv', index=False)
print("\nData successfully saved to nawy_properties.csv")

Scrolling iteration 1...
Properties loaded so far: 23
Scrolling iteration 2...
Properties loaded so far: 35
Scrolling iteration 3...
Properties loaded so far: 47
Scrolling iteration 4...
Properties loaded so far: 58
Scrolling iteration 5...
Properties loaded so far: 70
Scrolling iteration 6...
Properties loaded so far: 82
Scrolling iteration 7...
Properties loaded so far: 92
Scrolling iteration 8...
Properties loaded so far: 103
Scrolling iteration 9...
Properties loaded so far: 115
Scrolling iteration 10...
Properties loaded so far: 127
Scrolling iteration 11...
Properties loaded so far: 139
Scrolling iteration 12...
Properties loaded so far: 151
Scrolling iteration 13...
Properties loaded so far: 163
Scrolling iteration 14...
Properties loaded so far: 174
Scrolling iteration 15...
Properties loaded so far: 186
Scrolling iteration 16...
Properties loaded so far: 198
Scrolling iteration 17...
Properties loaded so far: 210
Scrolling iteration 18...
Properties loaded so far: 222
Scrollin

In [57]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
# url = "https://www.nawy.com/search?page_number=1&category=property"
url = "https://www.nawy.com/search?category=property&property_types=5"


async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container until no new properties load for 3 iterations
        previous_count = 0
        iteration = 1
        unchanged_count_strikes = 0

        while True:
            print(f"Scrolling iteration {iteration}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        // Scroll to the bottom of the container
                        container.scrollTo(0, container.scrollHeight);
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Wait 5 seconds to allow content to load

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

            # Check if new items were loaded
            if current_count == previous_count:
                unchanged_count_strikes += 1
                print(f"No new properties loaded. Strike {unchanged_count_strikes}/3")
            else:
                unchanged_count_strikes = 0  # Reset the strike counter if new items appeared

            # Break the loop if no new items were loaded for 5 continuous iterations
            if unchanged_count_strikes >= 5:
                print("All properties loaded (count unchanged for 5 iterations). Stopping scroll.")
                break

            previous_count = current_count
            iteration += 1

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('Chalet_nawy_properties.csv', index=False)
print("\nData successfully saved to Chalet_nawy_properties.csv")

Scrolling iteration 1...
Properties loaded so far: 23
Scrolling iteration 2...
Properties loaded so far: 34
Scrolling iteration 3...
Properties loaded so far: 46
Scrolling iteration 4...
Properties loaded so far: 58
Scrolling iteration 5...
Properties loaded so far: 70
Scrolling iteration 6...
Properties loaded so far: 82
Scrolling iteration 7...
Properties loaded so far: 93
Scrolling iteration 8...
Properties loaded so far: 105
Scrolling iteration 9...
Properties loaded so far: 117
Scrolling iteration 10...
Properties loaded so far: 129
Scrolling iteration 11...
Properties loaded so far: 141
Scrolling iteration 12...
Properties loaded so far: 152
Scrolling iteration 13...
Properties loaded so far: 164
Scrolling iteration 14...
Properties loaded so far: 176
Scrolling iteration 15...
Properties loaded so far: 188
Scrolling iteration 16...
Properties loaded so far: 200
Scrolling iteration 17...
Properties loaded so far: 211
Scrolling iteration 18...
Properties loaded so far: 223
Scrollin

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed



Data successfully saved to Chalet_nawy_properties.csv


In [53]:
df.duplicated().sum()

np.int64(0)

In [61]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
# url = "https://www.nawy.com/search?page_number=1&category=property"
url = "https://www.nawy.com/search?category=property&property_types=7"


async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container until no new properties load for 3 iterations
        previous_count = 0
        iteration = 1
        unchanged_count_strikes = 0

        while True:
            print(f"Scrolling iteration {iteration}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        // Scroll to the bottom of the container
                        container.scrollTo(0, container.scrollHeight);
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Wait 5 seconds to allow content to load

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

            # Check if new items were loaded
            if current_count == previous_count:
                unchanged_count_strikes += 1
                print(f"No new properties loaded. Strike {unchanged_count_strikes}/5")
            else:
                unchanged_count_strikes = 0  # Reset the strike counter if new items appeared

            # Break the loop if no new items were loaded for 5 continuous iterations
            if unchanged_count_strikes >= 5:
                print("All properties loaded (count unchanged for 5 iterations). Stopping scroll.")
                break

            previous_count = current_count
            iteration += 1

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('Townhouse_nawy_properties.csv', index=False)
print("\nData successfully saved to Townhouse_nawy_properties.csv")

Scrolling iteration 1...
Properties loaded so far: 23
Scrolling iteration 2...
Properties loaded so far: 33
Scrolling iteration 3...
Properties loaded so far: 45
Scrolling iteration 4...
Properties loaded so far: 57
Scrolling iteration 5...
Properties loaded so far: 69
Scrolling iteration 6...
Properties loaded so far: 81
Scrolling iteration 7...
Properties loaded so far: 93
Scrolling iteration 8...
Properties loaded so far: 105
Scrolling iteration 9...
Properties loaded so far: 117
Scrolling iteration 10...
Properties loaded so far: 129
Scrolling iteration 11...
Properties loaded so far: 140
Scrolling iteration 12...
Properties loaded so far: 152
Scrolling iteration 13...
Properties loaded so far: 164
Scrolling iteration 14...
Properties loaded so far: 176
Scrolling iteration 15...
Properties loaded so far: 188
Scrolling iteration 16...
Properties loaded so far: 200
Scrolling iteration 17...
Properties loaded so far: 212
Scrolling iteration 18...
Properties loaded so far: 224
Scrollin

In [60]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
# url = "https://www.nawy.com/search?page_number=1&category=property"
url = "https://www.nawy.com/search?category=property&property_types=1"


async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container until no new properties load for 3 iterations
        previous_count = 0
        iteration = 1
        unchanged_count_strikes = 0

        while True:
            print(f"Scrolling iteration {iteration}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        // Scroll to the bottom of the container
                        container.scrollTo(0, container.scrollHeight);
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Wait 5 seconds to allow content to load

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

            # Check if new items were loaded
            if current_count == previous_count:
                unchanged_count_strikes += 1
                print(f"No new properties loaded. Strike {unchanged_count_strikes}/5")
            else:
                unchanged_count_strikes = 0  # Reset the strike counter if new items appeared

            # Break the loop if no new items were loaded for 5 continuous iterations
            if unchanged_count_strikes >= 5:
                print("All properties loaded (count unchanged for 5 iterations). Stopping scroll.")
                break

            previous_count = current_count
            iteration += 1

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('Villa_nawy_properties.csv', index=False)
print("\nData successfully saved to Villa_nawy_properties.csv")

Scrolling iteration 1...
Properties loaded so far: 24
Scrolling iteration 2...
Properties loaded so far: 34
Scrolling iteration 3...
Properties loaded so far: 44
Scrolling iteration 4...
Properties loaded so far: 56
Scrolling iteration 5...
Properties loaded so far: 68
Scrolling iteration 6...
Properties loaded so far: 78
Scrolling iteration 7...
Properties loaded so far: 90
Scrolling iteration 8...
Properties loaded so far: 97
Scrolling iteration 9...
Properties loaded so far: 105
Scrolling iteration 10...
Properties loaded so far: 105
No new properties loaded. Strike 1/5
Scrolling iteration 11...
Properties loaded so far: 105
No new properties loaded. Strike 2/5
Scrolling iteration 12...
Properties loaded so far: 105
No new properties loaded. Strike 3/5
Scrolling iteration 13...
Properties loaded so far: 105
No new properties loaded. Strike 4/5
Scrolling iteration 14...
Properties loaded so far: 105
No new properties loaded. Strike 5/5
All properties loaded (count unchanged for 5 ite

In [62]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
# url = "https://www.nawy.com/search?page_number=1&category=property"
url = "https://www.nawy.com/search?category=property&property_types=8"


async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container until no new properties load for 3 iterations
        previous_count = 0
        iteration = 1
        unchanged_count_strikes = 0

        while True:
            print(f"Scrolling iteration {iteration}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        // Scroll to the bottom of the container
                        container.scrollTo(0, container.scrollHeight);
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Wait 5 seconds to allow content to load

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

            # Check if new items were loaded
            if current_count == previous_count:
                unchanged_count_strikes += 1
                print(f"No new properties loaded. Strike {unchanged_count_strikes}/5")
            else:
                unchanged_count_strikes = 0  # Reset the strike counter if new items appeared

            # Break the loop if no new items were loaded for 5 continuous iterations
            if unchanged_count_strikes >= 5:
                print("All properties loaded (count unchanged for 5 iterations). Stopping scroll.")
                break

            previous_count = current_count
            iteration += 1

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('Office_nawy_properties.csv', index=False)
print("\nData successfully saved to Office_nawy_properties.csv")

Scrolling iteration 1...
Properties loaded so far: 23
Scrolling iteration 2...
Properties loaded so far: 34
Scrolling iteration 3...
Properties loaded so far: 45
Scrolling iteration 4...
Properties loaded so far: 53
Scrolling iteration 5...
Properties loaded so far: 62
Scrolling iteration 6...
Properties loaded so far: 73
Scrolling iteration 7...
Properties loaded so far: 84
Scrolling iteration 8...
Properties loaded so far: 84
No new properties loaded. Strike 1/5
Scrolling iteration 9...
Properties loaded so far: 84
No new properties loaded. Strike 2/5
Scrolling iteration 10...
Properties loaded so far: 84
No new properties loaded. Strike 3/5
Scrolling iteration 11...
Properties loaded so far: 84
No new properties loaded. Strike 4/5
Scrolling iteration 12...
Properties loaded so far: 84
No new properties loaded. Strike 5/5
All properties loaded (count unchanged for 5 iterations). Stopping scroll.

Data successfully saved to Office_nawy_properties.csv


In [63]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
# url = "https://www.nawy.com/search?page_number=1&category=property"
url = "https://www.nawy.com/search?category=property&property_types=11"


async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container until no new properties load for 3 iterations
        previous_count = 0
        iteration = 1
        unchanged_count_strikes = 0

        while True:
            print(f"Scrolling iteration {iteration}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        // Scroll to the bottom of the container
                        container.scrollTo(0, container.scrollHeight);
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Wait 5 seconds to allow content to load

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

            # Check if new items were loaded
            if current_count == previous_count:
                unchanged_count_strikes += 1
                print(f"No new properties loaded. Strike {unchanged_count_strikes}/5")
            else:
                unchanged_count_strikes = 0  # Reset the strike counter if new items appeared

            # Break the loop if no new items were loaded for 5 continuous iterations
            if unchanged_count_strikes >= 5:
                print("All properties loaded (count unchanged for 5 iterations). Stopping scroll.")
                break

            previous_count = current_count
            iteration += 1

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('Cabin_nawy_properties.csv', index=False)
print("\nData successfully saved to Cabin_nawy_properties.csv")

Scrolling iteration 1...
Properties loaded so far: 24
Scrolling iteration 2...
Properties loaded so far: 36
Scrolling iteration 3...
Properties loaded so far: 48
Scrolling iteration 4...
Properties loaded so far: 60
Scrolling iteration 5...
Properties loaded so far: 72
Scrolling iteration 6...
Properties loaded so far: 74
Scrolling iteration 7...
Properties loaded so far: 74
No new properties loaded. Strike 1/5
Scrolling iteration 8...
Properties loaded so far: 74
No new properties loaded. Strike 2/5
Scrolling iteration 9...
Properties loaded so far: 74
No new properties loaded. Strike 3/5
Scrolling iteration 10...
Properties loaded so far: 74
No new properties loaded. Strike 4/5
Scrolling iteration 11...
Properties loaded so far: 74
No new properties loaded. Strike 5/5
All properties loaded (count unchanged for 5 iterations). Stopping scroll.

Data successfully saved to Cabin_nawy_properties.csv


In [64]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
# url = "https://www.nawy.com/search?page_number=1&category=property"
url = "https://www.nawy.com/search?category=property&property_types=3"


async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container until no new properties load for 3 iterations
        previous_count = 0
        iteration = 1
        unchanged_count_strikes = 0

        while True:
            print(f"Scrolling iteration {iteration}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        // Scroll to the bottom of the container
                        container.scrollTo(0, container.scrollHeight);
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Wait 5 seconds to allow content to load

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

            # Check if new items were loaded
            if current_count == previous_count:
                unchanged_count_strikes += 1
                print(f"No new properties loaded. Strike {unchanged_count_strikes}/5")
            else:
                unchanged_count_strikes = 0  # Reset the strike counter if new items appeared

            # Break the loop if no new items were loaded for 5 continuous iterations
            if unchanged_count_strikes >= 5:
                print("All properties loaded (count unchanged for 5 iterations). Stopping scroll.")
                break

            previous_count = current_count
            iteration += 1

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('Duplex_nawy_properties.csv', index=False)
print("\nData successfully saved to Duplex_nawy_properties.csv")

Scrolling iteration 1...
Properties loaded so far: 23
Scrolling iteration 2...
Properties loaded so far: 35
Scrolling iteration 3...
Properties loaded so far: 47
Scrolling iteration 4...
Properties loaded so far: 58
Scrolling iteration 5...
Properties loaded so far: 70
Scrolling iteration 6...
Properties loaded so far: 81
Scrolling iteration 7...
Properties loaded so far: 93
Scrolling iteration 8...
Properties loaded so far: 105
Scrolling iteration 9...
Properties loaded so far: 117
Scrolling iteration 10...
Properties loaded so far: 128
Scrolling iteration 11...
Properties loaded so far: 140
Scrolling iteration 12...
Properties loaded so far: 152
Scrolling iteration 13...
Properties loaded so far: 164
Scrolling iteration 14...
Properties loaded so far: 176
Scrolling iteration 15...
Properties loaded so far: 188
Scrolling iteration 16...
Properties loaded so far: 200
Scrolling iteration 17...
Properties loaded so far: 212
Scrolling iteration 18...
Properties loaded so far: 224
Scrollin

In [65]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
# url = "https://www.nawy.com/search?page_number=1&category=property"
url = "https://www.nawy.com/search?category=property&property_types=6"


async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container until no new properties load for 3 iterations
        previous_count = 0
        iteration = 1
        unchanged_count_strikes = 0

        while True:
            print(f"Scrolling iteration {iteration}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        // Scroll to the bottom of the container
                        container.scrollTo(0, container.scrollHeight);
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Wait 5 seconds to allow content to load

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

            # Check if new items were loaded
            if current_count == previous_count:
                unchanged_count_strikes += 1
                print(f"No new properties loaded. Strike {unchanged_count_strikes}/5")
            else:
                unchanged_count_strikes = 0  # Reset the strike counter if new items appeared

            # Break the loop if no new items were loaded for 5 continuous iterations
            if unchanged_count_strikes >= 5:
                print("All properties loaded (count unchanged for 5 iterations). Stopping scroll.")
                break

            previous_count = current_count
            iteration += 1

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('Twinhouse_nawy_properties.csv', index=False)
print("\nData successfully saved to Twinhouse_nawy_properties.csv")

Scrolling iteration 1...
Properties loaded so far: 21
Scrolling iteration 2...
Properties loaded so far: 32
Scrolling iteration 3...
Properties loaded so far: 43
Scrolling iteration 4...
Properties loaded so far: 54
Scrolling iteration 5...
Properties loaded so far: 65
Scrolling iteration 6...
Properties loaded so far: 77
Scrolling iteration 7...
Properties loaded so far: 87
Scrolling iteration 8...
Properties loaded so far: 99
Scrolling iteration 9...
Properties loaded so far: 108
Scrolling iteration 10...
Properties loaded so far: 108
No new properties loaded. Strike 1/5
Scrolling iteration 11...
Properties loaded so far: 108
No new properties loaded. Strike 2/5
Scrolling iteration 12...
Properties loaded so far: 108
No new properties loaded. Strike 3/5
Scrolling iteration 13...
Properties loaded so far: 108
No new properties loaded. Strike 4/5
Scrolling iteration 14...
Properties loaded so far: 108
No new properties loaded. Strike 5/5
All properties loaded (count unchanged for 5 ite

In [66]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
# url = "https://www.nawy.com/search?page_number=1&category=property"
url = "https://www.nawy.com/search?category=property&property_types=4"


async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container until no new properties load for 3 iterations
        previous_count = 0
        iteration = 1
        unchanged_count_strikes = 0

        while True:
            print(f"Scrolling iteration {iteration}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        // Scroll to the bottom of the container
                        container.scrollTo(0, container.scrollHeight);
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Wait 5 seconds to allow content to load

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

            # Check if new items were loaded
            if current_count == previous_count:
                unchanged_count_strikes += 1
                print(f"No new properties loaded. Strike {unchanged_count_strikes}/5")
            else:
                unchanged_count_strikes = 0  # Reset the strike counter if new items appeared

            # Break the loop if no new items were loaded for 5 continuous iterations
            if unchanged_count_strikes >= 5:
                print("All properties loaded (count unchanged for 5 iterations). Stopping scroll.")
                break

            previous_count = current_count
            iteration += 1

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('Penthouse_nawy_properties.csv', index=False)
print("\nData successfully saved to Penthouse_nawy_properties.csv")

Scrolling iteration 1...
Properties loaded so far: 23
Scrolling iteration 2...
Properties loaded so far: 33
Scrolling iteration 3...
Properties loaded so far: 44
Scrolling iteration 4...
Properties loaded so far: 56
Scrolling iteration 5...
Properties loaded so far: 68
Scrolling iteration 6...
Properties loaded so far: 79
Scrolling iteration 7...
Properties loaded so far: 91
Scrolling iteration 8...
Properties loaded so far: 102
Scrolling iteration 9...
Properties loaded so far: 113
Scrolling iteration 10...
Properties loaded so far: 125
Scrolling iteration 11...
Properties loaded so far: 137
Scrolling iteration 12...
Properties loaded so far: 149
Scrolling iteration 13...
Properties loaded so far: 161
Scrolling iteration 14...
Properties loaded so far: 173
Scrolling iteration 15...
Properties loaded so far: 185
Scrolling iteration 16...
Properties loaded so far: 197
Scrolling iteration 17...
Properties loaded so far: 209
Scrolling iteration 18...
Properties loaded so far: 219
Scrollin

In [67]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
# url = "https://www.nawy.com/search?page_number=1&category=property"
url = "https://www.nawy.com/search?category=property&property_types=10"


async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container until no new properties load for 3 iterations
        previous_count = 0
        iteration = 1
        unchanged_count_strikes = 0

        while True:
            print(f"Scrolling iteration {iteration}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        // Scroll to the bottom of the container
                        container.scrollTo(0, container.scrollHeight);
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Wait 5 seconds to allow content to load

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

            # Check if new items were loaded
            if current_count == previous_count:
                unchanged_count_strikes += 1
                print(f"No new properties loaded. Strike {unchanged_count_strikes}/5")
            else:
                unchanged_count_strikes = 0  # Reset the strike counter if new items appeared

            # Break the loop if no new items were loaded for 5 continuous iterations
            if unchanged_count_strikes >= 5:
                print("All properties loaded (count unchanged for 5 iterations). Stopping scroll.")
                break

            previous_count = current_count
            iteration += 1

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('Retail_nawy_properties.csv', index=False)
print("\nData successfully saved to Retail_nawy_properties.csv")

Scrolling iteration 1...
Properties loaded so far: 24
Scrolling iteration 2...
Properties loaded so far: 36
Scrolling iteration 3...
Properties loaded so far: 48
Scrolling iteration 4...
Properties loaded so far: 60
Scrolling iteration 5...
Properties loaded so far: 70
Scrolling iteration 6...
Properties loaded so far: 81
Scrolling iteration 7...
Properties loaded so far: 90
Scrolling iteration 8...
Properties loaded so far: 102
Scrolling iteration 9...
Properties loaded so far: 114
Scrolling iteration 10...
Properties loaded so far: 125
Scrolling iteration 11...
Properties loaded so far: 136
Scrolling iteration 12...
Properties loaded so far: 148
Scrolling iteration 13...
Properties loaded so far: 160
Scrolling iteration 14...
Properties loaded so far: 170
Scrolling iteration 15...
Properties loaded so far: 182
Scrolling iteration 16...
Properties loaded so far: 193
Scrolling iteration 17...
Properties loaded so far: 204
Scrolling iteration 18...
Properties loaded so far: 204
No new p

In [68]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
# url = "https://www.nawy.com/search?page_number=1&category=property"
url = "https://www.nawy.com/search?category=property&property_types=13"


async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container until no new properties load for 3 iterations
        previous_count = 0
        iteration = 1
        unchanged_count_strikes = 0

        while True:
            print(f"Scrolling iteration {iteration}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        // Scroll to the bottom of the container
                        container.scrollTo(0, container.scrollHeight);
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Wait 5 seconds to allow content to load

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

            # Check if new items were loaded
            if current_count == previous_count:
                unchanged_count_strikes += 1
                print(f"No new properties loaded. Strike {unchanged_count_strikes}/5")
            else:
                unchanged_count_strikes = 0  # Reset the strike counter if new items appeared

            # Break the loop if no new items were loaded for 5 continuous iterations
            if unchanged_count_strikes >= 5:
                print("All properties loaded (count unchanged for 5 iterations). Stopping scroll.")
                break

            previous_count = current_count
            iteration += 1

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('Clinic_nawy_properties.csv', index=False)
print("\nData successfully saved to Clinic_nawy_properties.csv")

Scrolling iteration 1...
Properties loaded so far: 23
Scrolling iteration 2...
Properties loaded so far: 34
Scrolling iteration 3...
Properties loaded so far: 45
Scrolling iteration 4...
Properties loaded so far: 57
Scrolling iteration 5...
Properties loaded so far: 68
Scrolling iteration 6...
Properties loaded so far: 80
Scrolling iteration 7...
Properties loaded so far: 92
Scrolling iteration 8...
Properties loaded so far: 103
Scrolling iteration 9...
Properties loaded so far: 114
Scrolling iteration 10...
Properties loaded so far: 125
Scrolling iteration 11...
Properties loaded so far: 137
Scrolling iteration 12...
Properties loaded so far: 149
Scrolling iteration 13...
Properties loaded so far: 161
Scrolling iteration 14...
Properties loaded so far: 173
Scrolling iteration 15...
Properties loaded so far: 185
Scrolling iteration 16...
Properties loaded so far: 197
Scrolling iteration 17...
Properties loaded so far: 209
Scrolling iteration 18...
Properties loaded so far: 221
Scrollin

In [69]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
# url = "https://www.nawy.com/search?page_number=1&category=property"
url = "https://www.nawy.com/search?category=property&property_types=9"


async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container until no new properties load for 3 iterations
        previous_count = 0
        iteration = 1
        unchanged_count_strikes = 0

        while True:
            print(f"Scrolling iteration {iteration}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        // Scroll to the bottom of the container
                        container.scrollTo(0, container.scrollHeight);
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Wait 5 seconds to allow content to load

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

            # Check if new items were loaded
            if current_count == previous_count:
                unchanged_count_strikes += 1
                print(f"No new properties loaded. Strike {unchanged_count_strikes}/5")
            else:
                unchanged_count_strikes = 0  # Reset the strike counter if new items appeared

            # Break the loop if no new items were loaded for 5 continuous iterations
            if unchanged_count_strikes >= 5:
                print("All properties loaded (count unchanged for 5 iterations). Stopping scroll.")
                break

            previous_count = current_count
            iteration += 1

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('Studio_nawy_properties.csv', index=False)
print("\nData successfully saved to Studio_nawy_properties.csv")

Scrolling iteration 1...
Properties loaded so far: 23
Scrolling iteration 2...
Properties loaded so far: 35
Scrolling iteration 3...
Properties loaded so far: 47
Scrolling iteration 4...
Properties loaded so far: 58
Scrolling iteration 5...
Properties loaded so far: 68
Scrolling iteration 6...
Properties loaded so far: 80
Scrolling iteration 7...
Properties loaded so far: 91
Scrolling iteration 8...
Properties loaded so far: 102
Scrolling iteration 9...
Properties loaded so far: 113
Scrolling iteration 10...
Properties loaded so far: 124
Scrolling iteration 11...
Properties loaded so far: 136
Scrolling iteration 12...
Properties loaded so far: 148
Scrolling iteration 13...
Properties loaded so far: 159
Scrolling iteration 14...
Properties loaded so far: 171
Scrolling iteration 15...
Properties loaded so far: 183
Scrolling iteration 16...
Properties loaded so far: 194
Scrolling iteration 17...
Properties loaded so far: 206
Scrolling iteration 18...
Properties loaded so far: 218
Scrollin

In [70]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
# url = "https://www.nawy.com/search?page_number=1&category=property"
url = "https://www.nawy.com/search?category=property&property_types=23"


async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container until no new properties load for 3 iterations
        previous_count = 0
        iteration = 1
        unchanged_count_strikes = 0

        while True:
            print(f"Scrolling iteration {iteration}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        // Scroll to the bottom of the container
                        container.scrollTo(0, container.scrollHeight);
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Wait 5 seconds to allow content to load

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

            # Check if new items were loaded
            if current_count == previous_count:
                unchanged_count_strikes += 1
                print(f"No new properties loaded. Strike {unchanged_count_strikes}/5")
            else:
                unchanged_count_strikes = 0  # Reset the strike counter if new items appeared

            # Break the loop if no new items were loaded for 5 continuous iterations
            if unchanged_count_strikes >= 5:
                print("All properties loaded (count unchanged for 5 iterations). Stopping scroll.")
                break

            previous_count = current_count
            iteration += 1

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('Loft_nawy_properties.csv', index=False)
print("\nData successfully saved to Loft_nawy_properties.csv")

Scrolling iteration 1...
Properties loaded so far: 24
Scrolling iteration 2...
Properties loaded so far: 36
Scrolling iteration 3...
Properties loaded so far: 48
Scrolling iteration 4...
Properties loaded so far: 60
Scrolling iteration 5...
Properties loaded so far: 72
Scrolling iteration 6...
Properties loaded so far: 84
Scrolling iteration 7...
Properties loaded so far: 87
Scrolling iteration 8...
Properties loaded so far: 87
No new properties loaded. Strike 1/5
Scrolling iteration 9...
Properties loaded so far: 87
No new properties loaded. Strike 2/5
Scrolling iteration 10...
Properties loaded so far: 87
No new properties loaded. Strike 3/5
Scrolling iteration 11...
Properties loaded so far: 87
No new properties loaded. Strike 4/5
Scrolling iteration 12...
Properties loaded so far: 87
No new properties loaded. Strike 5/5
All properties loaded (count unchanged for 5 iterations). Stopping scroll.

Data successfully saved to Loft_nawy_properties.csv


In [72]:
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import asyncio
import json
import pandas as pd

# Define the target URL for the Nawy property search page
# url = "https://www.nawy.com/search?page_number=1&category=property"
url = "https://www.nawy.com/search?category=property&property_types=43%2C47%2C40%2C12"


async def extract_properties():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the scrollable container (which holds the property listings) is loaded
        await page.wait_for_selector("div.sc-46853e00-0.dtcWKQ")

        # ================================
        # INFINITE SCROLL SIMULATION
        # ================================
        # Scroll down inside the scrollable container until no new properties load for 3 iterations
        previous_count = 0
        iteration = 1
        unchanged_count_strikes = 0

        while True:
            print(f"Scrolling iteration {iteration}...")
            await page.evaluate("""
                () => {
                    const container = document.querySelector('div.sc-46853e00-0.dtcWKQ');
                    if (container) {
                        // Scroll to the bottom of the container
                        container.scrollTo(0, container.scrollHeight);
                    }
                }
            """)
            await page.wait_for_timeout(5000)  # Wait 5 seconds to allow content to load

            # Count the number of property cards currently loaded in the DOM
            current_count = await page.locator("div.cover-image").count()
            print(f"Properties loaded so far: {current_count}")

            # Check if new items were loaded
            if current_count == previous_count:
                unchanged_count_strikes += 1
                print(f"No new properties loaded. Strike {unchanged_count_strikes}/5")
            else:
                unchanged_count_strikes = 0  # Reset the strike counter if new items appeared

            # Break the loop if no new items were loaded for 5 continuous iterations
            if unchanged_count_strikes >= 5:
                print("All properties loaded (count unchanged for 5 iterations). Stopping scroll.")
                break

            previous_count = current_count
            iteration += 1

        # ================================
        # EXTRACT FULL PAGE HTML AFTER SCROLLING
        # ================================
        # After scrolling, retrieve the complete page HTML (now includes dynamically loaded content)
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it
html_content = await extract_properties()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# List to hold all extracted property dictionaries
all_properties_data = []

# Find all cover images, then find their parent <a> tag to use as the root for each property card
cover_images = soup.find_all('div', class_='cover-image')

for cover in cover_images:
    card = cover.find_parent('a')
    if not card:
        continue

    extracted_data = {}

    # 1. URL Path
    if 'href' in card.attrs:
        extracted_data['url_path'] = f"https://www.nawy.com{card['href']}"
    else:
        extracted_data['url_path'] = None

    # 2. Tag (e.g., "Resale")
    tag_elem = card.find('div', class_='tag')
    extracted_data['tag'] = tag_elem.get_text(strip=True) if tag_elem else None

    # 3. Cover Image URL
    cover_image = card.find('img', alt='Cover Image')
    extracted_data['cover_image'] = cover_image['src'] if cover_image else None

    # 4. Location
    area_elem = card.find('div', class_='area')
    extracted_data['location'] = area_elem.get_text(strip=True) if area_elem else None

    # 5. Property Name & Compound
    name_elem = card.find('div', class_='name')
    extracted_data['property_name'] = name_elem.get_text(strip=True) if name_elem else None

    # 6. Developer Logo URL
    logo_image = card.find('img', alt='logo')
    extracted_data['developer_logo'] = logo_image['src'] if logo_image else None

    # 7. Full Title/Description
    title_elem = card.find('h2')
    extracted_data['title'] = title_elem.get_text(strip=True) if title_elem else None

    # Description
    desc_elem = card.select_one('h2.sc-4b9910fd-0.hHfZHY')
    extracted_data['description'] = desc_elem.get_text(strip=True) if desc_elem else None

    # 8. Property Features (Area, Beds, Baths)
    values = card.find_all('span', class_='value')
    labels = card.find_all('span', class_='label')

    for val, lbl in zip(values, labels):
        feature_name = lbl.get_text(strip=True)
        feature_value = val.get_text(strip=True)
        extracted_data[feature_name] = feature_value

    # 9. Payment Plan
    down_payment_elem = card.find('span', class_='down-payment')
    if down_payment_elem:
        raw_payment = down_payment_elem.get_text(separator=' ', strip=True)
        extracted_data['payment_plan'] = " ".join(raw_payment.split())
    else:
        extracted_data['payment_plan'] = None

    # 10. Price
    price_elem = card.find('span', class_='price')
    if price_elem:
        raw_price = price_elem.get_text(separator=' ', strip=True)
        extracted_data['price'] = " ".join(raw_price.split())
    else:
        extracted_data['price'] = None

    # Append the extracted dictionary to our list
    all_properties_data.append(extracted_data)

# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(all_properties_data)

# Display the first few rows in the notebook
# print(df.head())

# Save the DataFrame to a CSV file
df.to_csv('Family_House_Pharmacy_Building_Administrative_nawy_properties.csv', index=False)
print("\nData successfully saved to Family_House_Pharmacy_Building_Administrative_nawy_properties.csv")

Scrolling iteration 1...
Properties loaded so far: 21
Scrolling iteration 2...
Properties loaded so far: 32
Scrolling iteration 3...
Properties loaded so far: 43
Scrolling iteration 4...
Properties loaded so far: 54
Scrolling iteration 5...
Properties loaded so far: 65
Scrolling iteration 6...
Properties loaded so far: 75
Scrolling iteration 7...
Properties loaded so far: 87
Scrolling iteration 8...
Properties loaded so far: 99
Scrolling iteration 9...
Properties loaded so far: 111
Scrolling iteration 10...
Properties loaded so far: 122
Scrolling iteration 11...
Properties loaded so far: 134
Scrolling iteration 12...
Properties loaded so far: 146
Scrolling iteration 13...
Properties loaded so far: 158
Scrolling iteration 14...
Properties loaded so far: 169
Scrolling iteration 15...
Properties loaded so far: 181
Scrolling iteration 16...
Properties loaded so far: 193
Scrolling iteration 17...
Properties loaded so far: 204
Scrolling iteration 18...
Properties loaded so far: 204
No new pr

ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
playwright._impl._errors.TargetClosedError: Target page, context or browser has been closed



Data successfully saved to Family_House_Pharmacy_Building_Administrative_nawy_properties.csv
